# Lyapunov Spectrum: Ordered / Critical / Chaotic Regimes

Computes Lyapunov exponents via QR algorithm. Critical: max exponent λ₁ ≈ 0.



In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.utils.seed_registry import SeedRegistry
from src.utils.device_manager import DeviceManager

SeedRegistry.get_instance().set_master_seed(42)
device = DeviceManager.get_instance().get_device()
print(f'Device: {device}')


## Lyapunov Exponent Computation

In [ ]:
from src.core.lyapunov.lyapunov import analyze_lyapunov, detect_regime

import numpy as np

def generate_jacobians(n, depth, sigma_w):
    """Random Jacobians from Gaussian init."""
    rng = np.random.default_rng(42)
    return [rng.standard_normal((n, n)) * (sigma_w / n**0.5) for _ in range(depth)]

n, depth = 32, 100
results = {}
for sigma_w in [0.8, 1.4, 2.5]:
    Js = generate_jacobians(n, depth, sigma_w)
    res = analyze_lyapunov(Js)
    results[sigma_w] = res
    print(f"sigma_w={sigma_w:.1f}: MLE={res.mle:.4f}, regime={res.regime}, KY_dim={res.kaplan_yorke_dim:.1f}")


## MLE vs σ_w (Phase Transition)

In [ ]:
sigma_w_vals = np.linspace(0.5, 3.0, 40)
mle_vals = []
for sw in sigma_w_vals:
    Js = generate_jacobians(n=16, depth=50, sigma_w=sw)
    res = analyze_lyapunov(Js)
    mle_vals.append(res.mle)

plt.figure(figsize=(7,4))
plt.plot(sigma_w_vals, mle_vals, "o-", ms=4)
plt.axhline(0, color="r", ls="--", label="λ₁=0 (critical)")
plt.axvline(1.4, color="g", ls=":", label="σ_w=1.4 (paper)")
plt.xlabel("σ_w"); plt.ylabel("Max Lyapunov Exponent λ₁")
plt.title("Phase transition in Lyapunov spectrum"); plt.legend()
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
